# C07 · C2 groundwork: how far is the SAR signal from the footprints? (per tile)

Re-measures, inside the package, what the 31 Oct audit measured on a sample: for every tile of the
**development and secondary** events, the shift between the optical footprints and the SAR signal
(cross-correlation at 1/4 resolution, within ±40 m), how confident the match is, and whether it beats a
permutation null (the same footprints against 3 other tiles of the same event).

Why: Gate 1 for C2 should only count events where an offset exists (`docs/c2/feasibility_2026-11-20.md`).
This gives each event's median offset, its size in tokens, and how consistent its direction is.

* **Runtime:** CPU is enough. Secret: `GH_TOKEN`. Needs `01_data/derived/buildings.parquet` (p01).
* **Reads footprints and SAR only; no damage labels.** Confirmatory events are not touched.
* **Time:** about 15 minutes (BRIGHT post-event download ~5 min, offsets ~5–10 min).
* **Outputs:** `offsets_by_tile.csv`, `offsets_by_event.csv` in this run's folder and in `01_data/derived/c2_offsets/`.

In [ ]:
NOTEBOOK_NAME = "c07_c2_offsets"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# BRIGHT post-event SAR (~2 GB)
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Offsets per tile, development + secondary events (footprints only)
import pandas as pd
from damagesar_bench.c2_offsets import event_summary, measure_offsets
from damagesar_bench.encoders.extract import read_bright_tile
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet",
                    columns=["event", "role", "tile_id", "instance_id", "excluded_reason", "segmentation_px"])
EVENTS = sorted(B.loc[B["role"].isin(["development", "secondary"]), "event"].unique())
print("events:", EVENTS)
off = measure_offsets(B, lambda t: read_bright_tile(ROOT, t), EVENTS, factor=4, max_shift_m=40.0, n_null=3,
                      seed=0, log=print)
del B
OUT = f"{PROJECT}/01_data/derived/c2_offsets"
os.makedirs(OUT, exist_ok=True)
for d in (RUN, OUT):
    off.to_csv(f"{d}/offsets_by_tile.csv", index=False)
print(len(off), "tiles measured")

In [ ]:
# Per event: median offset, in tokens, confidence, null, direction consistency
from damagesar_bench.encoders.extract import read_bright_tile
summ = event_summary(off, peak_z_min=3.0)
# token side (ViT/16, native) from one tile per event
tok = {}
for ev, g in off.groupby("event"):
    _, geom = read_bright_tile(ROOT, g["tile_id"].iloc[0])
    tok[ev] = 16 * (geom.px_x_m + geom.px_y_m) / 2
summ["token_side_m"] = summ["event"].map(tok)
summ["median_offset_tokens"] = summ["median_offset_confident_m"] / summ["token_side_m"]
for d in (RUN, OUT):
    summ.to_csv(f"{d}/offsets_by_event.csv", index=False)
print(summ.round(2).to_string(index=False))
print("\nRead: an event is a C2 candidate if share_beats_null is high, the median offset is >= ~1 token, and")
print("direction_R is high (one consistent direction) or a per-tile shift is used.")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/checks/
import json
from google.colab import _message
nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
stamp = datetime.date.today().isoformat()
for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
               f"{PROJECT}/03_notebooks/checks/{stamp}_{NOTEBOOK_NAME}.ipynb"):
    with open(target, "w") as fh:
        json.dump(nb, fh)
    print("saved", target)
print("run folder:", sorted(os.listdir(RUN)))